In [1]:
import os
import pandas as pd
from pathlib import Path
import shutil


main_dir = Path.cwd().parent.parent
raw_data_dir = main_dir / 'data' / 'raw'
processed_data_dir = main_dir / 'data' / 'processed'


Import files and merge

In [2]:
sensing_df = pd.read_csv(raw_data_dir / 'sensing.csv')
steps_df = pd.read_csv(raw_data_dir / 'steps.csv')
merged_df = pd.merge(sensing_df, steps_df, on=['uid', 'day'], how='left')

ema_df = pd.read_csv(raw_data_dir / 'general_ema.csv')


Filter unnecessary columns

In [3]:

prefix_cols_to_keep = ('act_in_vehicle_', 'act_still_','act_on_bike_', 'act_running_', 'act_walking_', 'act_on_foot_', 'loc_dist_','loc_food_dur', 'loc_health_dur', 'loc_home_dur', 'loc_leisure_dur', 'loc_self_dorm_dur', 'loc_other_dorm_dur', 'loc_social_dur', 'loc_study_dur', 'loc_workout_dur', 'loc_worship_dur','audio_convo_duration_', 'call_in_duration_', 'call_in_num_', 'call_miss_num_', 'call_out_duration_', 'call_out_num_', 'other_playing_duration_', 'sleep_dur', 'unlock_duration_', 'unlock_num_', 'step_')


base_cols = ['uid', 'is_ios', 'day']

filtered_cols = [col for col in merged_df.columns if (col in base_cols or col.startswith(prefix_cols_to_keep)) and not col.endswith(('ep_1', 'ep_2', 'ep_3'))]
filtered_df = merged_df[filtered_cols].copy()

foot_hr_col_names = []
running_hr_col_names = []
walking_hr_col_names = [] 

for hour in range(24):
    foot_hr_col_names.append(f'act_on_foot_hr_{hour}')
    running_hr_col_names.append(f'act_running_hr_{hour}')
    walking_hr_col_names.append(f'act_walking_hr_{hour}')

    filtered_df[foot_hr_col_names[hour]] = filtered_df[[foot_hr_col_names[hour], running_hr_col_names[hour], walking_hr_col_names[hour]]].sum(axis=1, min_count=1)
    

filtered_df = filtered_df.drop(columns=walking_hr_col_names + running_hr_col_names)

filtered_df['act_on_foot_ep_0'] = filtered_df[['act_on_foot_ep_0', 'act_walking_ep_0', 'act_running_ep_0']].sum(axis=1, min_count=1)

filtered_df = filtered_df.drop(columns=['act_walking_ep_0', 'act_running_ep_0'])

datetimes = pd.to_datetime(filtered_df['day'].astype(str), format='%Y%m%d')

filtered_df['day'] = datetimes.dt.strftime('%Y-%m-%d')




Create a folder for each user, separating daily and hourly data

N.B. Only the most active 5 iOS users and 5 Android users are processed for testing reasons:
    'Most active' is defined by counting the non-null values across some key columns ('act_still_ep_0', 'unlock_duration_ep_0', 'sleep_duration', 'loc_dist_ep_0')


In [4]:
users_dir = processed_data_dir / 'users'
if users_dir.exists():
    shutil.rmtree(users_dir)
users_dir.mkdir(exist_ok=True)

num_users_to_include_per_platform = 5


key_columns_coverage_assessment = ['act_still_ep_0', 'unlock_duration_ep_0', 'sleep_duration', 'loc_dist_ep_0' ]

user_data_coverage = filtered_df.groupby(['uid', 'is_ios'])[key_columns_coverage_assessment].count().sum(axis=1).reset_index(name='active_stats')



ios_uids = user_data_coverage[user_data_coverage['is_ios'] == 1].sort_values(by='active_stats', ascending=False).head(num_users_to_include_per_platform)['uid'].to_list()
android_uids = user_data_coverage[user_data_coverage['is_ios'] == 0].sort_values(by='active_stats', ascending=False).head(num_users_to_include_per_platform)['uid'].to_list()


uids_to_process = ios_uids + android_uids

batch_df = filtered_df[filtered_df['uid'].isin(uids_to_process)]




hourly_col_names = ['act_in_vehicle_hr_', 'act_still_hr_', 'act_on_bike_hr_', 'act_on_foot_hr_', 'audio_convo_duration_hr_', 'loc_dist_hr_', 'other_playing_duration_hr_', 'unlock_duration_hr_', 'unlock_num_hr_', 'step_hr_']

hourly_cols = ['uid', 'day'] + [col for col in filtered_df.columns if '_hr_' in col]

daily_cols = ['uid', 'day'] + [col for col in filtered_df.columns if col not in hourly_cols]


for (uid, user_data) in batch_df.groupby('uid'):

    
    current_user_dir = users_dir / uid
    current_user_dir.mkdir(exist_ok=True)

    hourly_user_data = user_data[hourly_cols].copy()
    daily_user_data = user_data[daily_cols].copy()


    melted_hourly_user_data = pd.wide_to_long(
        df=hourly_user_data,
        stubnames=hourly_col_names,
        i=['uid', 'day'],
        j='hour'
    ).reset_index()

    

    hourly_col_name_dict = {col: col.replace('_hr_', '') for col in melted_hourly_user_data.columns}
    melted_hourly_user_data = melted_hourly_user_data.rename(columns=hourly_col_name_dict).sort_values(by=['day', 'hour'])

    daily_col_name_dict = {col: col.replace('_ep_0', '') for col in daily_user_data.columns}
    daily_user_data = daily_user_data.rename(columns=daily_col_name_dict).sort_values(by='day')

    daily_user_data.to_csv(current_user_dir / 'daily.csv', index=False)
    melted_hourly_user_data.to_csv(current_user_dir / 'hourly.csv', index=False)


    
    


EMA surveys data processing

In [5]:
ema_df = ema_df.dropna(axis=0,subset=['pam','phq4-1','phq4-2','phq4-3','phq4-4','phq4_resp_mean','phq4_resp_median','phq4_score','social_level','sse3-1','sse3-2','sse3-3','sse3-4','sse3_resp_mean','sse3_resp_median','stress','avg_ema_spent_time'])

cols_to_drop = [col for col in ema_df.columns if 'time' in col or 'resp' in col]
ema_df = ema_df.drop(columns=cols_to_drop)

datetimes = pd.to_datetime(ema_df['day'].astype(str), format='%Y%m%d')
ema_df['day'] = datetimes.dt.strftime('%Y-%m-%d')


In [6]:
for (uid, user_data) in ema_df[ema_df['uid'].isin(uids_to_process)].groupby('uid'):
    current_user_dir = users_dir / uid
    user_data.to_csv(current_user_dir / 'ema.csv', index=False)